In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from abc import ABC, abstractmethod
from pathlib import Path
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from typing import Optional
import copy

In [2]:
class XANESWeightedLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=1.0, gamma=0.1):
        '''
        alpha, beta, gamma - коэффициенты в комбинированном лоссе (mse, mae, mape) соотвевтственно.
        '''
        super().__init__()
        self.alpha, self.beta, self.gamma = alpha, beta, gamma
        self.mse = nn.MSELoss()
        self.mae = nn.L1Loss()

    def forward(self, pred, target):
        loss_mse = self.mse(pred, target)
        loss_mae = self.mae(pred, target)
        # Стабилизация MAPE для дробных CN
        denom = torch.clamp(torch.abs(target), min=1e-2)
        loss_mape = torch.mean(torch.abs((target - pred) / denom))
        return self.alpha * loss_mse + self.beta * loss_mae + self.gamma * loss_mape

class BaseModelWrapper(ABC):
    def __init__(self, params=None):
        self.model = None
        self.params = params

    @abstractmethod
    def fit(self, X_train, y_train, X_val, y_val):
        '''Возвращает (train_history, val_history)'''
        pass

    @abstractmethod
    def predict(self, X):
        '''Возвращает предсказания модели'''
        pass

    @abstractmethod
    def get_feature_importance(self, X_sample=None):
        '''Возвращает массив важности признаков'''
        pass

In [3]:
class XANESVisualizer:
    @staticmethod
    def plot_learning_curves(train_losses:np.ndarray, 
                             val_losses:Optional[np.ndarray]=None, 
                             title:str='Learning Curve', 
                             save_root:Optional[str|Path]=None):
        if not train_losses: return
        plt.figure(figsize=(8, 4))
        plt.plot(train_losses, label='Train Loss', color='#1f77b4', lw=2)
        if val_losses:
            plt.plot(val_losses, label='Val Loss', color='#ff7f0e', lw=2)
        plt.title(title)
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.yscale('log')
        plt.legend()
        plt.grid(True, which="both", ls="-", alpha=0.2)
        if save_root:
            t_path = Path(save_root,'learning_curve.png')
            plt.savefig(t_path, dpi=600, bbox_inches='tight')
        plt.show()
        
    @staticmethod
    def plot_evaluation_board(y_true:np.ndarray, 
                              y_pred:np.ndarray,
                              y_test_true:np.ndarray,
                              y_test_pred:np.ndarray,
                              cn_label:str='CN', 
                              save_root:Optional[str|Path]=None):
        '''Выводит Parity для Train/Test и распределение ошибок (Residuals)'''
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Parity Train
        axes[0].scatter(y_true, y_pred, alpha=0.5, s=15, color='gray')
        axes[0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')
        
        limits_tr = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        residuals_tr = y_true - y_pred
        std_dev_tr = np.std(residuals_tr)
        x_range_tr = np.linspace(limits_tr[0], limits_tr[1], 100)
        # 1 std (68% данных при нормальном распределении)
        axes[0].fill_between(x_range_tr, x_range_tr - std_dev_tr, x_range_tr + std_dev_tr, 
                         color='green', alpha=0.2, label=f'1σ ({std_dev_tr:.3f})')

        # 2 std (95% данных)
        axes[0].fill_between(x_range_tr, x_range_tr - 2*std_dev_tr, x_range_tr + 2*std_dev_tr, 
                         color='green', alpha=0.1, label=f'2σ ({2*std_dev_tr:.3f})')

        
        axes[0].set_title(f'{cn_label} Parity (Train)')
        axes[0].set_xlabel('True')
        axes[0].set_ylabel('Predicted')
        axes[0].legend(loc='upper left')
        
        # Parity Test
        axes[1].scatter(y_test_true, y_test_pred, alpha=0.6, s=20, color='#2c3e50')
        axes[1].plot([y_test_true.min(), y_test_true.max()], [y_test_true.min(), y_test_true.max()], 'r--')

        limits_te = [min(y_test_true.min(), y_test_pred.min()), max(y_test_true.max(), y_test_pred.max())]
        residuals_te = y_test_true - y_test_pred
        std_dev_te = np.std(residuals_te)
        x_range_te = np.linspace(limits_te[0], limits_te[1], 100)
        # 1 std (68% данных при нормальном распределении)
        axes[1].fill_between(x_range_te, x_range_te - std_dev_te, x_range_te + std_dev_te, 
                         color='green', alpha=0.2, label=f'1σ ({std_dev_te:.3f})')

        # 2 std (95% данных)
        axes[1].fill_between(x_range_te, x_range_te - 2*std_dev_te, x_range_te + 2*std_dev_te, 
                         color='green', alpha=0.1, label=f'2σ ({2*std_dev_te:.3f})')

        
        axes[1].set_title(f'{cn_label} Parity (Test)')
        axes[1].set_xlabel('True')
        axes[1].set_ylabel('Predicted')
        axes[1].legend(loc='upper left')
    
        # Residuals Distribution (Гистограмма ошибок)
        residuals = y_test_pred.flatten() - y_test_true.flatten()
        sns.histplot(residuals, kde=True, ax=axes[2], color='#e74c3c', bins=30)
        axes[2].axvline(x=0, color='black', linestyle='--')
        axes[2].set_title(f'{cn_label} Residuals Distribution (Test)')
        axes[2].set_xlabel('Error (Pred - True)')
        axes[2].set_ylabel('Frequency')
    
        plt.tight_layout()
        if save_root:
            t_path = Path(save_root,f'{cn_label}_evaluation_board.png')
            plt.savefig(t_path, dpi=600, bbox_inches='tight')
        plt.show()

    @staticmethod
    def plot_feature_importance_(spectrum:np.ndarray,
                                 importance:np.ndarray,
                                 title:str='Feature Importance',
                                 save_root:Optional[str|Path]=None):
        plt.figure(figsize=(10, 3))
        x = np.arange(len(spectrum))
        norm_imp = (importance - importance.min()) / (importance.max() - importance.min() + 1e-9)
        plt.plot(x, spectrum / (spectrum.max() + 1e-9), color='black', alpha=0.2, label='Spectrum')
        plt.fill_between(x, norm_imp, color='teal', alpha=0.7, label='Importance')
        plt.title(title)
        plt.legend()
        if save_root:
            t_path = Path(save_root,'feature_importance.png')
            plt.savefig(t_path, dpi=600, bbox_inches='tight')
        plt.show()

    @staticmethod
    def plot_feature_importance(spectrum:np.ndarray,
                                importance:np.ndarray,
                                energy_grid:np.ndarray=None, 
                                title:str='Feature Importance Map', 
                                save_root:Optional[str|Path]=None):
        '''
        spectrum: 1D array (средний спектр или конкретный образец)
        importance: 1D array той же длины
        energy_grid: опционально массив значений энергии (eV) для оси X
        '''
        if energy_grid is not None:
            if energy_grid.shape[0]!=spectrum.shape[0]:
                print('energy_grid.shape[0]!=spectrum.shape[0]')
                energy_grid = np.arange(len(spectrum))
        else:
            energy_grid = np.arange(len(spectrum))
            
        imp_norm = (importance - importance.min()) / (importance.max() - importance.min() + 1e-8)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), 
                                       gridspec_kw={'height_ratios': [4, 0.5]}, 
                                       sharex=True)
        
        ax1.plot(energy_grid, spectrum, color='black', lw=1.5, label='Mean Spectrum')
        ax1.set_title(title)
        ax1.set_ylabel('Intensity (normalized)')
        ax1.grid(True, alpha=0.3)
        
        # создаем двумерный массив из 1D вектора для imshow
        heatmap_data = imp_norm.reshape(1, -1)
        im = ax2.imshow(heatmap_data, aspect='auto', cmap='hot', 
                         extent=[energy_grid.min(), energy_grid.max(), 0, 1])
        
        ax2.set_yticks([]) # Убираем ось Y для хитмапы
        ax2.set_xlabel('Energy / Grid Point Index')
        
        # Добавляем цветовой индикатор
        cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(im, cax=cbar_ax, label='Relative Importance')
        
        plt.subplots_adjust(hspace=0.05)
        if save_root:
            t_path = Path(save_root,'feature_importance_map.png')
            plt.savefig(t_path, dpi=600, bbox_inches='tight')
        plt.show()

In [4]:
class XANESDataLoader:
    def __init__(self, standardize:bool=True):
        self.scaler = StandardScaler() if standardize else None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def load_data(self, paths: dict):
        '''
        paths: {'train_x': '...', 'train_y': '...', 'test_x': '...', 'test_y': '...'}
        '''
        data = {k: np.load(v)['arr_0'] for k, v in paths.items()}
        
        # Разделяем на X и y
        X_train, y_train = data['train_x'], data['train_y']
        X_test, y_test = data['test_x'], data['test_y']

        if self.scaler:
            X_train = self.scaler.fit_transform(X_train)
            X_test = self.scaler.transform(X_test)

        return X_train, X_test, y_train, y_test

In [5]:
class XANESPipeline:
    def __init__(self,
                 loader:XANESDataLoader,
                 visualizer:XANESVisualizer,
                 wrapper_instance: BaseModelWrapper,
                 e_grid: Optional[np.ndarray] = None
                ):
        '''
        loader: экземпляр загрузчика данных
        visualizer: экземпляр визуализатора
        wrapper_instance: инициализированный объект ModelWrapper (CatBoostWrapper, LinRegWrapper и т.д.)
        e_grid: сетка энергий для спектра
        '''
        self.loader = loader
        self.viz = visualizer
        self.wrapper_template = wrapper_instance  # Шаблон модели
        self.e_grid = e_grid
        self.results_summary = []

    def run(self, 
            paths:dict, 
            save_root:str = None,
            seed:int=42):
        '''
        paths: словарь с путями к данным {'train_x': '...', 'train_y': '...', 'test_x': '...', 'test_y': '...'}
        save_root: путь до корневой папки для сохранения всех графиков и отчетов
        '''
        X_full, X_test, y_full, y_test = self.loader.load_data(paths)
        X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.15, random_state=seed)
        
        mean_spec = X_train.mean(axis=0)

        for i in range(y_train.shape[1]):
            cn_label = f"CN{i+1}"
            print(f"\n🚀 Моделирование {cn_label}...")

            current_save_path = None
            if save_root:
                current_save_path = Path(save_root, cn_label)
                current_save_path.mkdir(parents=True, exist_ok=True)

            # Клонируем обертку, чтобы сбросить состояние модели перед новым обучением
            wrapper = copy.deepcopy(self.wrapper_template)
            
            t_hist, v_hist = wrapper.fit(X_train, y_train[:, i], X_val, y_val[:, i])

            train_preds = wrapper.predict(X_train)
            val_preds = wrapper.predict(X_val)
            test_preds = wrapper.predict(X_test)

            # Визуализация с автоматическим сохранением
            self.viz.plot_learning_curves(train_losses = t_hist,
                                          val_losses = v_hist, 
                                          title=f"History: {cn_label}", 
                                          save_root = current_save_path
                                         )
            
            self.viz.plot_feature_importance(spectrum = mean_spec, 
                                             importance = wrapper.get_feature_importance(mean_spec), 
                                             title=f"Importance: {cn_label}", 
                                             save_root=current_save_path,
                                             energy_grid=self.e_grid)

            self.viz.plot_evaluation_board(y_true = y_train[:, i],
                                           y_pred = train_preds, 
                                           y_test_true = y_test[:, i],
                                           y_test_pred = test_preds, 
                                           cn_label=cn_label, 
                                           save_root=current_save_path
                                          )


            # Сбор метрик
            self._record_metrics(cn_label, { 'train': (y_train[:, i], train_preds),
                                             'val'  : (y_val[:, i], val_preds), 
                                             'test' : (y_test[:, i], test_preds)
                                           }
                                )

        self._print_final_report(save_root)

    def _record_metrics(self, label:str, data_splits:dict):
        for split_name, (true, pred) in data_splits.items():
            true, pred = np.asarray(true).flatten(), np.asarray(pred).flatten()
            
            self.results_summary.append({'Target': label, 
                                         'Split': split_name, 
                                         'R2': r2_score(true, pred), 
                                         'MAE': mean_absolute_error(true, pred), 
                                         'RMSE': np.sqrt(mean_squared_error(true, pred))
                                        })

    def _print_final_report(self, save_root:Optional[Path|str]=None):
        df = pd.DataFrame(self.results_summary)
        print('\n' + '='*65 + '\n📊 ИТОГОВЫЙ ОТЧЕТ\n' + '='*65)
        
        pivot = df.pivot(index='Target', columns='Split', values=['R2', 'MAE'])
        report_str = pivot.to_string(float_format=lambda x: f"{x:.4f}")
        print(report_str)
        
        if save_root:
            df.to_csv(Path(save_root, 'final_metrics.csv'), index=False)